# 02 · MF-Rényi extraction

Computes the MF-Rényi multifractal descriptors over the 14 segments of every
painting, in two scale bands, and stores them in `mfrenyi_b1` and `mfrenyi_b2`.

Each segment is covered by a regular grid of s × s boxes and three masses are
measured per box: **intensity sum** (`sum_`), **variance** (`var_`) and
**Shannon entropy** (`ent_`). Each mass yields its own partition function,
D_q and f(α), so every segment produces 13 × 3 = 39 descriptors.

| Band | Scale range | Ladder density | Columns |
|---|---|---|---|
| `b1` | 6 px to 25% of the side | √2 | 546 |
| `b2` | 25% to 75% of the side | 2^(1/4) | 546 |

**The run is resumable.** A row in the table means that painting is done, so if
the run is interrupted just execute the extraction cell again: it picks up where
it stopped. Failures are recorded in `extraction_log` with their message and do
not stop the sweep.

Rough cost: well under a second of computation per painting; the run time is
dominated by streaming and decoding the images.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import pandas as pd

from datasets import load_dataset

import utils as ut
import mfrenyi
import db

## Parameters

Same q range and scale bands as notebook 01, so the two methods describe the
same range of structures. Integers are pixels, floats are fractions of the
segment side.

In [ ]:
MAX_SIZE = 1380          # side of the normalised canvas
COMMIT_EVERY = 50        # paintings per transaction
REPORT_EVERY = 100       # paintings per progress line
LIMIT = None             # None processes everything; a number caps the run

BAND_B1 = dict(q_min=-5.0, q_max=5.0, s_min=6,    s_max=0.25, degree_scales=1)
BAND_B2 = dict(q_min=-5.0, q_max=5.0, s_min=0.25, s_max=0.75, degree_scales=2)

METHOD, BAND_TAG = "mfrenyi", "b1b2"     # keys in extraction_log

## Schema

The Rényi tables must hold the three masses (547 columns each). An older
schema with only 13 descriptors per segment is rebuilt here, **but only if the
table is empty**; a table with rows and the wrong columns stops the notebook
instead of being dropped.

In [ ]:
con = db.connect()
db.create_extraction_log(con)

builders = {"mfrenyi_b1": (db.create_mfrenyi_b1, "b1"),
            "mfrenyi_b2": (db.create_mfrenyi_b2, "b2")}

for table, (build, band) in builders.items():
    expected = db.feature_columns(db.FEATURES_RENYI, band)
    current = [r[1] for r in con.execute(f'PRAGMA table_info("{table}")')][1:]

    if current == expected:
        print(f"{table:<12} ok ({len(current)} feature columns)")
        continue

    n_rows = con.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0] if current else 0
    if n_rows:
        raise RuntimeError(f"{table} has {n_rows} rows and a different schema; "
                           "rebuild it by hand if that is intended")
    build(con, drop=True)

## Current state

In [ ]:
total = con.execute("SELECT COUNT(*) FROM metadata").fetchone()[0]
done = db.done_painting_ids(con, "mfrenyi_b1") & db.done_painting_ids(con, "mfrenyi_b2")
failed = db.failed_painting_ids(con, METHOD, BAND_TAG)
pending = sorted(set(range(total)) - done)

print(f"paintings in metadata : {total}")
print(f"already processed     : {len(done)}")
print(f"failed before         : {len(failed)}")
print(f"pending               : {len(pending)}")
print(f"first pending id      : {pending[0] if pending else None}")

## Scale ladder check

Box counting needs many boxes at every scale: a scale covered by a single box
gives p = 1, so χ(q) = 1 for every q and that point carries no information
about the painting. The table lists, for the largest segment (`seg11`) and the
smallest one (`seg31`), how many boxes each scale of each band produces.

The geometry is the same for every painting, so this check does not depend on
the image.

In [ ]:
rows = []
for name_seg, side in [("seg11", MAX_SIZE), ("seg31", MAX_SIZE // 3)]:
    for band, params in [("b1", BAND_B1), ("b2", BAND_B2)]:
        scales = mfrenyi.scale_ladder((side, side), params["s_min"],
                                      params["s_max"], params["degree_scales"])
        boxes = [(side // s) ** 2 for s in scales]
        rows.append(dict(segment=name_seg, band=band, n_scales=len(scales),
                         s_px=f"{scales[0]} to {scales[-1]}",
                         boxes=f"{boxes[0]} to {boxes[-1]}",
                         scales_single_box=sum(b == 1 for b in boxes)))

pd.DataFrame(rows)

## Per-painting extraction

Returns the two feature dictionaries with the column names already assembled,
for example `var_D1/seg23/b2`. If one segment fails, its features are left
missing instead of discarding the whole painting.

In [ ]:
def extract_one(img_norm):
    # MF-Rényi over the 14 segments, in both bands
    feats_b1, feats_b2 = {}, {}
    failures = []

    for grid in range(1, 4):
        for pos, seg in enumerate(ut.segment_image(img_norm, grid_size=grid)):
            name_seg = f"seg{grid}{pos + 1}"

            for band, params, feats in [("b1", BAND_B1, feats_b1),
                                        ("b2", BAND_B2, feats_b2)]:
                try:
                    _, f = mfrenyi.mf_renyi_features(seg, **params)
                    feats.update({f"{k}/{name_seg}/{band}": v for k, v in f.items()})
                except Exception as e:
                    failures.append(f"{band}/{name_seg}: {e}")

    return feats_b1, feats_b2, failures

## Single-painting smoke test

Before launching the long run: check that the column names match the schema,
how many features come out, and how many are NaN. The first call also pays the
numba compilation, so the time is measured on a second call.

D0 of the intensity sum should sit close to 2 for any painting without black
regions; a value far from 2 points at a problem in the scale ladder.

In [ ]:
ds_test = load_dataset("huggan/wikiart", split="train",
                      streaming=True, columns=["image"])
item = next(iter(ds_test))

img_norm = ut.normalize_image(img=np.array(item["image"]),
                              max_size=MAX_SIZE, gray=True)
extract_one(img_norm)                      # numba compilation

t0 = time.time()
f1, f2, failures = extract_one(img_norm)
dt = time.time() - t0

for band, f in [("b1", f1), ("b2", f2)]:
    expected = set(db.feature_columns(db.FEATURES_RENYI, band))
    obtained = set(f)
    print(f"[{band}] features           : {len(f)}")
    print(f"[{band}] NaN                : {sum(np.isnan(v) for v in f.values())}")
    print(f"[{band}] missing columns    : {sorted(expected - obtained)[:5]}")
    print(f"[{band}] unexpected columns : {sorted(obtained - expected)[:5]}")
    print(f"[{band}] sum_D0 on seg11    : {f[f'sum_D0/seg11/{band}']:.3f}")

print(f"\nseconds per painting : {dt:.2f}")
print(f"failed segments      : {failures[:3]}")

## Full extraction

This is the long cell. It can be interrupted with the stop button without
losing committed work: the `finally` block closes the pending transaction.

The stream starts at the first pending painting, so a resumed run does not
decode again every painting already stored. To resume, run the state cell
above and then this one again.

In [ ]:
start = pending[0] if pending else total
dataset = load_dataset("huggan/wikiart", split="train",
                       streaming=True, columns=["image"]).skip(start)

n_ok = n_err = n_skip = 0
t0 = time.time()

try:
    for painting_id, item in enumerate(dataset, start=start):
        if painting_id in done:
            n_skip += 1
            continue
        if LIMIT is not None and n_ok + n_err >= LIMIT:
            break

        try:
            img_norm = ut.normalize_image(img=np.array(item["image"]),
                                          max_size=MAX_SIZE, gray=True)
            feats_b1, feats_b2, failures = extract_one(img_norm)

            db.insert_features(con, "mfrenyi_b1", painting_id, feats_b1, commit=False)
            db.insert_features(con, "mfrenyi_b2", painting_id, feats_b2, commit=False)
            db.log_extraction(con, painting_id, METHOD, BAND_TAG,
                              "ok" if not failures else "error",
                              error=None if not failures else "; ".join(failures),
                              commit=False)
            n_ok += 1

        except Exception as e:
            db.log_extraction(con, painting_id, METHOD, BAND_TAG,
                              "error", error=e, commit=False)
            n_err += 1

        if (n_ok + n_err) % COMMIT_EVERY == 0:
            con.commit()

        if (n_ok + n_err) % REPORT_EVERY == 0:
            elapsed = time.time() - t0
            rate = (n_ok + n_err) / elapsed
            left = len(pending) - (n_ok + n_err)
            print(f"{n_ok + n_err:>6} processed  "
                  f"{n_err:>4} failed  "
                  f"{rate:.2f} paintings/s  "
                  f"~{left / rate / 3600:.1f} h to go")

finally:
    con.commit()
    print(f"\ncommitted: {n_ok} ok, {n_err} failed, "
          f"{n_skip} skipped as already done")

## Verification

How many rows landed, how many failures remain, and where the missing values
concentrate.

In [ ]:
pd.read_sql_query(f'''
    SELECT status, COUNT(*) AS paintings
    FROM extraction_log
    WHERE method = '{METHOD}'
    GROUP BY status
''', con)

In [ ]:
pd.read_sql_query(f'''
    SELECT painting_id, substr(error, 1, 120) AS error
    FROM extraction_log
    WHERE method = '{METHOD}' AND status = 'error'
    LIMIT 10
''', con)

### Missing values per feature

A high count on one column points at a systematic problem, for example a
regression that breaks on the small segments of band 2.

In [ ]:
def null_counts(table, band):
    cols = db.feature_columns(db.FEATURES_RENYI, band)
    parts = ", ".join(f'COUNT(*) - COUNT("{c}") AS "{c}"' for c in cols)
    nulls = pd.read_sql_query(f'SELECT {parts} FROM "{table}"', con).T
    nulls.columns = ["nulls"]
    return nulls.sort_values("nulls", ascending=False)

null_counts("mfrenyi_b2", "b2").head(14)

### Near-constant features

A descriptor that barely changes between paintings carries no information for
the classifier. Expected suspects: `sum_D0`, which only depends on the grid
geometry unless the painting has black regions, and the band 2 descriptors if
the ladder check above showed scales with a single box. Computed on a sample of
5000 paintings.

In [ ]:
def lowest_spread(table, n=5000, top=15):
    df = pd.read_sql_query(f'SELECT * FROM "{table}" ORDER BY RANDOM() LIMIT {n}',
                           con).drop(columns="painting_id")
    return (df.std().sort_values().head(top)
              .rename_axis("feature").reset_index(name="std"))

pd.concat({"b1": lowest_spread("mfrenyi_b1"),
           "b2": lowest_spread("mfrenyi_b2")}, axis=1)

In [ ]:
con.close()